# Day 4 — Type Hints & Pydantic Configuration Modeling

## Objective
Demonstrate configuration validation using Python type hints, Dataclasses, fixed-choice `Enum`s, and Pydantic v2 runtime schema enforcement. Compare static checking (`mypy`) vs runtime validation (`Pydantic`).

## 1. Type Hints & Schema Imports

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from pydantic import ValidationError
from task_analytics import (
    PipelineConfig, SimplePipelineConfig, PipelineMode, Device, GenericConfigContainer
)
print('Type hint and Pydantic components loaded successfully.')

Type hint and Pydantic components loaded successfully.


## 2. Valid Configuration Demonstration

Instantiate a valid `PipelineConfig` model and inspect field coercion and attributes.

In [2]:
config = PipelineConfig(
    data_path=Path('data'),
    batch_size=64,
    feature_columns=['title', 'priority'],
    mode=PipelineMode.TRAIN,
    device=Device.CPU,
    threshold=0.85,
)
print('Valid PipelineConfig loaded:')
print('  data_path:      ', config.data_path, type(config.data_path))
print('  batch_size:     ', config.batch_size, type(config.batch_size))
print('  mode:           ', config.mode, type(config.mode))
print('  threshold:      ', config.threshold)

Valid PipelineConfig loaded:
  data_path:       data <class 'pathlib.WindowsPath'>
  batch_size:      64 <class 'int'>
  mode:            PipelineMode.TRAIN <enum 'PipelineMode'>
  threshold:       0.85


## 3. Dataclass vs Pydantic Comparison

Standard `@dataclass` structures data but **does NOT enforce runtime validation**.

In [3]:
ds_config = SimplePipelineConfig(
    data_path=Path('non_existent_path'),
    batch_size=-100,
    mode='invalid_mode'
)
print('Dataclass accepted invalid batch_size=-100 without error:')
print('  ', ds_config)

Dataclass accepted invalid batch_size=-100 without error:
   SimplePipelineConfig(data_path=WindowsPath('non_existent_path'), batch_size=-100, mode='invalid_mode', threshold=0.5)


## 4. Multi-Scenario Validation Error Demonstrations (Pydantic)

Demonstrate Pydantic catching invalid configurations at runtime.

In [4]:
print('--- Scenario 1: Wrong Type (batch_size="large") ---')
try:
    PipelineConfig(data_path=Path('data'), batch_size='large')
except ValidationError as e:
    print(e)

print('\n--- Scenario 2: Out of Range Value (threshold=1.5) ---')
try:
    PipelineConfig(data_path=Path('data'), threshold=1.5)
except ValidationError as e:
    print(e)

print('\n--- Scenario 3: Non-existent Data Path ---')
try:
    PipelineConfig(data_path=Path('non_existent_folder/data.csv'))
except ValidationError as e:
    print(e)

print('\n--- Scenario 4: Missing Required Field ---')
try:
    PipelineConfig(batch_size=32)  # missing data_path
except ValidationError as e:
    print(e)

--- Scenario 1: Wrong Type (batch_size="large") ---
1 validation error for PipelineConfig
batch_size
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='large', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

--- Scenario 2: Out of Range Value (threshold=1.5) ---
1 validation error for PipelineConfig
threshold
  Value error, Configuration failed in PipelineConfig: threshold must be between 0.0 and 1.0, got 1.5. [type=value_error, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

--- Scenario 3: Non-existent Data Path ---
1 validation error for PipelineConfig
data_path
  Value error, Configuration failed in PipelineConfig: data_path must point to an existing path on disk: 'non_existent_folder\data.csv'. [type=value_error, input_value=WindowsPath('non_existent_folder/data.csv'), input_type=WindowsPath]
    For further inf

## 5. Generic Container Demonstration (`Generic[T]`)

Demonstrate generic type hints using `GenericConfigContainer`.

In [5]:
container = GenericConfigContainer[PipelineConfig](item=config, metadata={'env': 'staging'})
print('Generic Container item type:', type(container.get_item()).__name__)
print('Container metadata:', container.metadata)

Generic Container item type: PipelineConfig
Container metadata: {'env': 'staging'}


## Conclusion & Key Takeaways

- **Static vs Runtime**: Type hints and `mypy` verify types statically; Pydantic enforces validation at runtime.
- **Dataclasses vs Pydantic**: Use dataclasses for internal DTOs; use Pydantic for external configuration and API inputs.